# Worked solutions · Section 6 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of the section,
as F1–F5.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
Theorem 1.3.43 (`thm.dassiosZhao`), eq. (1.3.41, `eq.dzDraws`).
Type 0 is the market buys and type 1 the market sells.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebook `06-simulating-hawkes` runs them and verifies every key.

In [ ]:
from math import exp, log, sqrt

# Section 6 · Simulating a Hawkes process

**F1. Branching matrix and column sums.**
For the exponential kernel $\Gamma = A/\beta$, eq. (1.3.12, `eq.GammaIsAOverBeta`).
The column sum $\sum_e A_{e,e'}$ is the jump of the total intensity $\lambda_{\mathfrak g}$
caused by one event of type $e'$;
the column sum $\sum_e \Gamma_{e,e'}$ is the expected number of its direct offspring.
They differ by the factor $\beta$, Remark 1.3.10 (`remark.countsAgainstRates`).

**F2. Branching ratio.**
$\rho$ is the spectral radius of $\Gamma$ (Section 1.3.2 (`sec.hawkesProcesses`), after eq. (1.3.9, `eq.branchingMatrix`)).
Lemma 1.3.17 (`lemma.perronFrobenius`), part iii, applied to $\Gamma^\top$
bounds $\rho$ between the smallest and the largest column sum of $\Gamma$
(the paragraph after the lemma):
constant column sums $c$ give $\rho = c$.

**F3. Total rate.**
Under $\rho < 1$ and $\mu > 0$, Theorem 1.3.19 (`thm.stationarity`) gives
$$\lambda^* = (I - \Gamma)^{-1}\mu,$$
eq. (1.3.18, `eq.stationaryIntensity`).
The total rate is $\bar\lambda = \mathbf 1^\top\lambda^*$
and the total immigration rate $\bar\mu = \mathbf 1^\top\mu$
(the definitions after Remark 1.3.20 (`remark.strictBaseline`)).
When $\mathbf 1^\top\Gamma = \rho\,\mathbf 1^\top$,
Corollary 1.3.25 (`corol.constantClusterSize`), with eq. (1.3.22, `eq.clusterSize`), gives $\mathbf 1^\top(I-\Gamma)^{-1} = \mathbf 1^\top/(1-\rho)$, hence
$$\bar\lambda = \mathbf 1^\top(I-\Gamma)^{-1}\mu = \frac{\bar\mu}{1-\rho}.$$
This is eq. (1.3.23, `eq.meanClusterSize`), $\bar\lambda = \bar\mu\,\bar C$, with $\bar C = 1/(1-\rho)$.
From $Z = 0$ the mean count over $(0, T]$ falls short of $T\bar\lambda$ by an amount bounded in $T$,
eq. (1.3.33, `eq.forwardMeanCountsDeviation`) at $z = 0$;
one path over a horizon long against the relaxation time $1/(\beta(1-\rho))$ of Definition 1.3.36 (`def.relaxationTime`)
counts close to $\bar\lambda$ events per second, the process being ergodic
(the subsection on sub- and supercriticality of Section 1.3.4 (`sec.stability`)).

**F4. Intensity and state.**
$\lambda = \mu + AZ$, eq. (1.3.13, `eq.intensityFromState`),
so $\lambda_{\mathfrak g} = \mathbf 1^\top\lambda = \bar\mu + (\mathbf 1^\top A)\,Z$.
The state $Z$ of eq. (1.3.11, `eq.decayedCounts`) sums over the events strictly before $t$,
so it is left-continuous:
at an arrival $T_n$ of type $E_n$, $Z(T_n)$ is the value before the jump, the one the intensity uses,
and $Z(T_n+) = Z(T_n) + \mathbf e_{E_n}$ the value carried forward, Remark 1.3.41 (`remark.prePostJump`).
Between events the state only decays, eq. (1.3.38, `eq.update_of_exponential_intensity`):
$$Z(T_{n+1}) = \big(Z(T_n) + \mathbf e_{E_n}\big)\,e^{-\beta(T_{n+1} - T_n)}.$$

**F5. The exact scheme.**
Between events $\lambda_{\mathfrak g}(t) = \bar\mu + D\,e^{-\beta(t - T_n)}$,
with $D = \lambda_{\mathfrak g}(T_n+) - \bar\mu = (\mathbf 1^\top A)\,Z(T_n+) \geq 0$,
eq. (1.3.39, `eq.totalIntensityBetweenEvents`).
By Theorem 1.3.43 (`thm.dassiosZhao`), with $U_1, U_2$ independent and uniform on $(0,1)$,
$$\tau_1 = -\frac{\log U_1}{\bar\mu},
\qquad
\tau_2 = -\frac{1}{\beta}\log\Big(1 + \frac{\beta\log U_2}{D}\Big),$$
eq. (1.3.41, `eq.dzDraws`), with $\tau_2 = +\infty$ when $D = 0$ or $1 + \beta\log U_2/D \leq 0$;
the wait is $\tau_1 \wedge \tau_2$,
and the type is drawn at the arrival, before the jump,
with probabilities $\lambda_e(T_{n+1})/\lambda_{\mathfrak g}(T_{n+1})$, eq. (1.3.42, `eq.dzType`).
Algorithm 1.3.2 (`algo.dassiosZhao`) orders the steps:
the wait, the decay of the state to the arrival, the type, the jump.

## 6.1 · `question.thinningBoundAfterTheJump`

The self-exciting pair, $\mu = (0.25, 0.25)$,
$A = \begin{pmatrix} 2.8 & 0.2 \\ 0.2 & 2.8 \end{pmatrix}$, $\beta = 4$,
over $2\times10^4$ s, where the standard error of the rate is $0.04$.

**Answer: (D).** A prints a number close to 2, and B a number clearly below 2.

**The total rate.**
$\Gamma = A/4$ has both column sums $(2.8 + 0.2)/4 = 0.75$ (F1), so $\rho = 0.75$ (F2),
and with $\bar\mu = 0.5$ (F3)
$$\bar\lambda = \frac{\bar\mu}{1-\rho} = \frac{0.5}{0.25} = 2.$$

In [ ]:
rho = (2.8 + 0.2) / 4
mu_bar = 0.25 + 0.25
rho, mu_bar / (1 - rho)

**The bound.**
Thinning, Algorithm 1.3.1 (`algo.ogata`), draws candidates at the rate of a bound $b$
(written $\bar\lambda$ in the algorithm; here $\bar\lambda$ is the total rate)
and keeps each with probability $\lambda_{\mathfrak g}/b$.
It is exact when $b$ dominates $\lambda_{\mathfrak g}$ up to the next accepted candidate.
The paragraph on thinning that precedes the algorithm, in Section 1.3.5 (`sec.simulation`), supplies the bound:
$A \geq 0$ makes $\lambda_{\mathfrak g}$ non-increasing between events,
so the post-jump value $\lambda_{\mathfrak g}(T_n+)$ dominates it up to $T_{n+1}$;
the pre-jump value is smaller by $(\mathbf 1^\top A)_{E_n}$, here 3, and does not.

- **A** sets `bound = mu_bar + jumps @ state` after the jump, that is $b = \lambda_{\mathfrak g}(T_n+)$,
  and keeps it through the rejections.
  A bound never tightened costs rejections and nothing else:
  A is exact, and prints a number within a few standard errors of 2.
- **B** sets `bound = intensity` at every candidate.
  After a rejection this is $\lambda_{\mathfrak g}$ at the candidate,
  which dominates what follows, as in Algorithm 1.3.1 (`algo.ogata`).
  After an acceptance it is $\lambda_{\mathfrak g}(T_n)$, read before the jump,
  3 below $\lambda_{\mathfrak g}(T_n+)$.
  While $\lambda_{\mathfrak g} > b$ the test `rng.random() * bound <= intensity` always passes,
  so events arrive at the rate $b$ and not at the rate $\lambda_{\mathfrak g}$:
  the excitation of each accepted event is never seen in full.
  From a quiet state, $\lambda_{\mathfrak g} \approx \bar\mu = 0.5$,
  an event raises $\lambda_{\mathfrak g}$ to 3.5 while B keeps proposing at 0.5,
  and its next candidate comes after a wait $W$ of mean 2 s,
  over which the excitation keeps on average $\mathbb E[e^{-4W}] = 0.5/(0.5 + 4) = 1/9$ of itself.
  B's bound stays far below $\lambda_{\mathfrak g}(T_n+)$, and B prints a number clearly below 2.
  The verification notebook prints 0.72.

**The other choices.**
- (A) Both close to 2: takes `bound = intensity` after an acceptance for the first step of Algorithm 1.3.1 (`algo.ogata`),
  which sets the bound to $\lambda_{\mathfrak g}(T_n+)$; `intensity` is read before the jump.
- (B) A below, B close: exchanges the two alternatives. A's bound is loose, not low.
- (C) B clearly above: takes the lower bound to raise the acceptance probability $\lambda_{\mathfrak g}/b$,
  missing that the candidates then come at the lower rate $b$:
  B's events arrive at the rate $\min(b, \lambda_{\mathfrak g}) \leq \lambda_{\mathfrak g}$.

## 6.2 · `question.recordedStatesAliasTheJump`

A buy, a sell and a buy at $0.5$, $1.0$ and $1.5$ s;
$\beta = 2\log 2$, so that $e^{-\beta \cdot 0.5} = 1/2$.

**Answer: (C).**
A prints `[[1.0, 0.0], [0.5, 1.0], [1.25, 0.5]]`,
and B prints `[[1.25, 0.5], [1.25, 0.5], [1.25, 0.5]]`.

**What should be recorded.**
By eq. (1.3.38, `eq.update_of_exponential_intensity`) (F4) the pre-jump states are
$$Z(0.5) = (0, 0),
\qquad Z(1.0) = \tfrac12\big((0,0) + \mathbf e_0\big) = (0.5, 0),
\qquad Z(1.5) = \tfrac12\big((0.5,0) + \mathbf e_1\big) = (0.25, 0.5),$$
and the post-jump states $Z(T_n+) = Z(T_n) + \mathbf e_{E_n}$ are
$(1, 0)$, $(0.5, 1)$ and $(1.25, 0.5)$.
Remark 1.3.41 (`remark.prePostJump`) keeps the two apart.

**What is recorded.**
`record.append(state)` stores a reference to the array, not a copy of it.
- In A, `state = state * ...` binds `state` to a new array at every event.
  The record holds that array,
  and `state[event_type] += 1.0` then adds the jump to it, in place:
  each entry is the post-jump state of its own event.
- In B, `state *= ...` decays, in place, the one array made by `np.zeros(2)`.
  The record holds three references to it,
  and prints its final value, $(1.25, 0.5)$, three times.

Recording `state.copy()` gives the pre-jump states.

In [ ]:
half = exp(-2 * log(2) * 0.5)
half, (half * (0 + 1), half * 0), (half * 0.5, half * (0 + 1))

**The other choices.**
- (A) A records the pre-jump states:
  misses that the jump is added, in place, to the array A has just recorded.
- (B) Both record the pre-jump states: misses the aliasing in both.
- (D) Both record the post-jump states: misses that `*=` keeps one array through the loop.

## 6.3 · `question.intensityByRecursionAgainstSum`

$\lambda_{\mathfrak g}(T_n)$ at every event, by the sum over the past (A) and by the update (B),
timed on the first $n$ events of a path, $n = 1000, \dots, 16000$.

**Answer: (A).**
The time per event of B stays about the same and that of A grows with $n$:
B is faster at every $n$, by a factor that grows with $n$.

**The cost.**
A evaluates eq. (1.3.11, `eq.decayedCounts`) directly:
at the event of index $k$, the loop's `n`, the weights `np.exp(-params.decay * (now - times[:n]))` are $k$ terms,
one per earlier event,
so a path of $n$ events costs $\sum_{k<n} k = n(n-1)/2$ terms, $O(n^2)$,
and the time per event grows linearly in $n$.
B carries the state by eq. (1.3.38, `eq.update_of_exponential_intensity`),
a fixed number of operations per event, $O(1)$.
The sentence after eq. (1.3.38, `eq.update_of_exponential_intensity`) states both costs.

Vectorising A's sum lowers its constant and not its growth.
At $n = 1000$ A's body already makes more numpy calls than B's, over up to a thousand entries rather than two,
so B is faster from the first rung; the growth of A shows as $n$ increases.

**The other choices.**
- (B) Both constant per event: misses that A's sum runs over the whole past.
- (C) A faster, its body being one vectorised expression:
  one vectorised expression over $n$ terms is still $n$ terms.
- (D) Both grow, since each loops over the events:
  the outer loop is common to both and gives the factor $n$ of the path,
  not a growth of the time per event; only the body of A grows.

## 6.4 · `question.exactWaitOnGivenUniforms`

The asymmetric pair, $\mu = (0.2, 0.6)$,
$A = \begin{pmatrix} 1.2 & 4.8 \\ 0.4 & 0.8 \end{pmatrix}$, $\beta = 4$,
from $Z(0+) = (1.5, 1.0)$.
$U_1 = e^{-0.4}$; $U_2 = e^{-1}$ in the first case and $e^{-3}$ in the second;
the third uniform, $0.75$, goes to the type draw.

**Answer: (D).** `0.173`, then `0.5`.

**The quantities of F5.**
$\bar\mu = 0.8$, and the column sums of $A$ are $\mathbf 1^\top A = (1.6, 5.6)$, so
$$D = (\mathbf 1^\top A)\,Z(0+) = 1.6 \cdot 1.5 + 5.6 \cdot 1.0 = 8,$$
the excess of $\lambda_{\mathfrak g}(0+) = 8.8$ over $\bar\mu$, which is `excess` in the code.
By eq. (1.3.41, `eq.dzDraws`), $-\log U_1 = 0.4$ gives $\tau_1 = 0.4/0.8 = 0.5$ in both cases.

**First case, $\log U_2 = -1$.**
$1 + \beta\log U_2/D = 1 - 4/8 = 1/2 > 0$, so
$$\tau_2 = -\tfrac14\log\tfrac12 = \tfrac14\log 2 \approx 0.173,$$
and the wait is $\tau_1 \wedge \tau_2 = 0.173$.

**Second case, $\log U_2 = -3$.**
$1 - 12/8 = -1/2 \leq 0$, so $\tau_2 = +\infty$ by the convention of Theorem 1.3.43 (`thm.dassiosZhao`):
the excitation clock never fires, Remark 1.3.44 (`remark.defectiveClock`).
The wait is $\tau_1 = 0.5$.

`from_state` moves the clock to zero (`unito26.lob.hawkes._HawkesState.from_state`), so the printed `time` is the wait.

In [ ]:
mu_bar = 0.2 + 0.6
D = (1.2 + 0.4) * 1.5 + (4.8 + 0.8) * 1.0
tau_1 = 0.4 / mu_bar
remaining_1 = 1 + 4 * (-1) / D
remaining_2 = 1 + 4 * (-3) / D
D, tau_1, remaining_1, -log(remaining_1) / 4, remaining_2

**The other choices.**
- (A) `0.5`, then `0.5`:
  the column sums of $\Gamma = A/4$, $(0.4, 1.4)$, taken for those of $A$ (F1).
  Then $D = 0.4 \cdot 1.5 + 1.4 \cdot 1.0 = 2$, and $1 - 4/2 = -1$ and $1 - 12/2 = -5$ are both negative:
  $\tau_2 = +\infty$ twice, and the wait is $\tau_1 = 0.5$ twice.
- (B) `0.5`, then `inf`: the later of the two clocks taken for the earlier,
  $\tau_1 \vee \tau_2$ for $\tau_1 \wedge \tau_2$: $0.5 \vee 0.173 = 0.5$, then $0.5 \vee \infty = \infty$.
- (C) `0.173`, then `0.32`: $-\log U_1$ multiplied by $\bar\mu$ rather than divided by it,
  $0.4 \cdot 0.8 = 0.32$; in the first case the excitation clock still fires first.
- (E) `0.173`, then `inf`: the infinite $\tau_2$ returned as the wait, the minimum with $\tau_1$ forgotten.

In [ ]:
D_gamma = 0.4 * 1.5 + 1.4 * 1.0
D_gamma, 1 + 4 * (-1) / D_gamma, 0.4 * mu_bar

## 6.5 · `question.typeDrawnBeforeTheDecay`

The state and the uniforms of the first case of 6.4.
`DrawFirst` draws the type before `_advance`.

**Answer: (E).** `1 [0.75, 1.5]`, then `0 [1.25, 0.5]`.

**The package.**
The wait of 6.4 is $\tau_2 = \tfrac14\log 2$,
and `_advance` multiplies the state by $e^{-\beta\tau_2} = 1 + \beta\log U_2/D = 1/2$:
$Z(T_1) = (0.75, 0.5)$.
The type is drawn on the intensities at the arrival, before the jump, eq. (1.3.42, `eq.dzType`) (F5),
with $\lambda = \mu + AZ$, eq. (1.3.13, `eq.intensityFromState`):
$$\lambda(T_1) =
\begin{pmatrix} 0.2 + 1.2 \cdot 0.75 + 4.8 \cdot 0.5 \\ 0.6 + 0.4 \cdot 0.75 + 0.8 \cdot 0.5 \end{pmatrix}
= \begin{pmatrix} 3.5 \\ 1.3 \end{pmatrix}.$$
The cumulative sums are $(3.5, 4.8)$ and the threshold is $0.75 \cdot 4.8 = 3.6$.
`np.searchsorted` returns the first index at which the cumulative sum reaches 3.6, which is 1:
a market sell, and the jump gives $Z(T_1+) = (0.75, 1.5)$.

In [ ]:
z = (0.75, 0.5)
lam = (0.2 + 1.2 * z[0] + 4.8 * z[1], 0.6 + 0.4 * z[0] + 0.8 * z[1])
lam, lam[0] + lam[1], 0.75 * (lam[0] + lam[1])

**`DrawFirst`.**
It draws on the intensities at $Z(0+) = (1.5, 1.0)$, those left by the last event:
$$\mu + A\,Z(0+) = (0.2 + 1.8 + 4.8,\; 0.6 + 0.6 + 0.8) = (6.8, 2.0).$$
The cumulative sums are $(6.8, 8.8)$ and the threshold is $0.75 \cdot 8.8 = 6.6 \leq 6.8$: a market buy.
The jump gives $(2.5, 1.0)$,
and `_advance` then halves the state with its own jump in it, to $(1.25, 0.5)$.
The wait is unchanged, since `excess` is computed before the draw.

Algorithm 1.3.2 (`algo.dassiosZhao`) orders the steps as the package does:
the decay to the arrival, the draw on $\lambda(T_{n+1})$, the jump.

In [ ]:
z = (1.5, 1.0)
lam = (0.2 + 1.2 * z[0] + 4.8 * z[1], 0.6 + 0.4 * z[0] + 0.8 * z[1])
lam, lam[0] + lam[1], 0.75 * (lam[0] + lam[1])

**The other choices.**
- (A) `1 [0.75, 1.5]` on both lines:
  reads `DrawFirst` as the package, missing that its draw comes before the decay.
- (B) `0 [1.75, 0.5]` on the second line:
  the buy of `DrawFirst` with its jump added after the halving, $(0.75 + 1, 0.5)$;
  the code adds the jump first, and halves it with the state.
- (C) `0 [2.5, 1.0]` on the second line: the state of `DrawFirst` read before `_advance`.
- (D) `0 [1.75, 0.5]` on the first line: the threshold taken as the uniform, 0.75, without the factor `cumulative[-1]`;
  it falls among the buys in both classes, and the second line is unchanged.

## 6.6 · `question.clippedExcitationClockAddsEvents`

$\mu = (0.5, 0.5)$, $A = \begin{pmatrix} 15 & 10 \\ 10 & 15 \end{pmatrix}$, $\beta = 50$,
over $2\times10^4$ s, where the standard error of the rate is $0.02$.
`Clipped` replaces $1 + \beta\log U_2/D$ by its maximum with $10^{-12}$.

**Answer: (B).**
`ExponentialHawkes` prints a number close to 2, and `Clipped` a number clearly above 2.

**The total rate.**
The column sums of $A$ are 25, so those of $\Gamma = A/50$ are $0.5 = \rho$ (F1, F2),
and $\bar\lambda = \bar\mu/(1-\rho) = 1/0.5 = 2$ (F3).

**The defective clock.**
By F5 and the proof of Theorem 1.3.43 (`thm.dassiosZhao`), $\tau_2 = +\infty$ exactly when $-\log U_2 \geq D/\beta$,
an event of probability $e^{-D/\beta}$:
the excited factor of eq. (1.3.40, `eq.dzSurvival`) carries the finite mass $D/\beta$,
the expected number of direct offspring the state still owes, Remark 1.3.44 (`remark.defectiveClock`).
Every step after the first starts just after an event, where $\mathbf 1^\top Z(T_n+) \geq 1$, so $D \geq 25$
and this probability is at most $e^{-1/2} \approx 0.61$, nearly attained after a long quiet spell;
the first step, from $Z = 0$, has $D = 0$ and $\tau_2 = +\infty$ in both classes, `Clipped` keeping the branch `excess <= 0.0`.
The package then takes the wait $\tau_1$, and the next event is an immigrant.

**The clipped clock.**
`Clipped` fires every such clock at $-\log(10^{-12})/\beta \approx 27.63/50 \approx 0.553$ s.
It fires before the baseline clock whenever $\tau_1 > 0.553$,
an event of probability $e^{-\bar\mu \cdot 0.553} = e^{-0.553} \approx 0.58$.
Each such firing is an event the model does not have, and it raises the intensity,
so it calls forth further events.

In [ ]:
rho = (15 + 10) / 50
clip = -log(1e-12) / 50
rho, 1 / (1 - rho), exp(-25 / 50), clip, exp(-1.0 * clip)

**The other choices.**
- (A) Both close to 2: treats $\tau_2 = +\infty$ as a numerical edge case,
  which Remark 1.3.44 (`remark.defectiveClock`) says it is not: clipping it biases the immigrant share.
- (C) `Clipped` below 2: reads `max` as `min`, which would put every firing of the excitation clock at 0.553 s;
  as written, the clip only brings forward the clocks that would never fire, and each such firing adds an event.
- (D) The package below 2, its clock failing to fire:
  an infinite $\tau_2$ loses no event; the wait is then $\tau_1$,
  and $\tau_1 \wedge \tau_2$ has the law eq. (1.3.40, `eq.dzSurvival`).

## 6.7 · `question.selfAndCrossExcitingShareTheirTimes`

The self-exciting pair, $A = \begin{pmatrix} 2.8 & 0.2 \\ 0.2 & 2.8 \end{pmatrix}$,
and the cross-exciting pair, $A = \begin{pmatrix} 0.2 & 2.8 \\ 2.8 & 0.2 \end{pmatrix}$,
with $\mu = (0.25, 0.25)$ and $\beta = 4$, simulated on seed 3 over 100 s.

**Answer: (C).** `True False`.

**The times.**
Both matrices have column sums $(3, 3)$,
so under both $\lambda_{\mathfrak g} = 0.5 + 3\,\mathbf 1^\top Z$ (F4).
By eq. (1.3.11, `eq.decayedCounts`), $\mathbf 1^\top Z(t) = \sum_{T_n < t} e^{-\beta(t - T_n)}$
is a function of the times of the past events and not of their types.
So $D = 3\,\mathbf 1^\top Z(T_n+)$, and the waits of eq. (1.3.41, `eq.dzDraws`) (F5)
are the same functions of the past times under both kernels.
Each step draws two uniforms for the wait and one for the type, under both,
so the two generators of seed 3 stay in step:
the same uniforms give the same waits,
and the times agree up to floating-point rounding,
since `_total_jump @ decayed_counts` sums different components on the two paths.
`np.allclose` prints `True`.

**The types.**
The type is drawn on the full intensities $\mu + AZ$, eq. (1.3.42, `eq.dzType`), where the two matrices differ.
By Remark 1.3.45 (`remark.typeDraw`) the wait sees $A$ only through $\mathbf 1^\top A$,
and the type draw is where the full matrix re-enters.
Over 100 s, some 200 events in expectation ($\bar\lambda = 2$), the two type sequences differ: `False`.

**The other choices.**
- (A) `True True`: misses that the type draw sees the full matrix.
- (B) `False False`: takes the waits to see the matrix beyond its column sums.
- (D) `ValueError`: takes the self-exciting pair, whose buys call forth buys, for the one with more events;
  the two have the same waits, hence the same number of events.

## 6.8 · `question.eventsHoldTheCrossingEvent`

A simulator of the one-way pair, $\bar\mu = 0.5$, seed 11, run to 10 s and then to 40 s,
against a single run to 40 s.

**Answer: (A).** `True 0`, then `False 1`.

`self.step()` draws the next event and, in drawing it,
moves the clock, decays the state, adds the jump (Algorithm 1.3.2 (`algo.dassiosZhao`))
and consumes three uniforms.
The first event past 10 s is drawn in this way before its time is compared with the horizon.
- The package stores it in `self._crossing` and returns.
  The second call yields it first, since it lies before 40 s,
  and continues with `step()` from the state after it.
  The two stretches are the single run: `True 0`.
- `Dropping` returns without storing it.
  Its second call continues with `step()` from the same clock and state,
  which already include the dropped event,
  so every later event is the one the single run draws, and only the dropped one is missing:
  `False 1`.

In both, the second call continues from where the first stopped, not from time zero.
The second line needs an event in $(10, 40]$:
without one, the event `Dropping` drops lies past 40 s, the single run does not have it either,
and the line reads `True 0`.
Seed 11 has one: thirty seconds without an event have a probability below $e^{-30\bar\mu} = e^{-15}$,
by eq. (1.3.40, `eq.dzSurvival`).

**The other choices.**
- (B) `True 0` on both lines: takes the dropped event for undrawn,
  as though the clock, the state and the generator had not moved.
- (C) `False 1` on both lines: misses that the package holds the crossing event.
- (D) `False` and a negative number: reads the second call as restarting from time zero,
  which would give the two stretches more events than the single run.

## 6.9 · `question.seedsOfAnOrderFlowMonteCarlo`

The weakly exciting pair, of baselines $(0.9, 0.9)$ in `unito26.lob.config.WEAKLY_EXCITING_PAIR_PARAMS`:
$\bar\mu = 1.8$ and $\bar\lambda = 2$ (F3); paths of 10 s.
The quoted line keeps a `Generator` as given and builds a fresh one from anything else.

**Answer: (B).** `True False False`.

- `path(5) == path(5)`: each call builds `np.random.default_rng(5)`,
  so both paths are drawn from the start of one stream: `True`.
- `path(rng) == path(rng)`: the two simulators share one generator,
  and the second reads the part of its stream after the part the first consumed.
  Two different paths, both non-empty:
  a path is empty with probability $e^{-10\bar\mu} = e^{-18}$, by eq. (1.3.40, `eq.dzSurvival`) at $D = 0$.
  `False`.
- `path(5) == path(rng)`: the arguments of `print` are evaluated from left to right,
  so this `path(rng)` is the third use of `rng`,
  which has moved past the numbers with which `default_rng(5)` starts: `False`.

Had `path(rng)` been called first, it would have equalled `path(5)`,
since `rng` is `default_rng(5)`.
Paths that are independent and reproducible are drawn from one generator,
seeded once and passed to every simulator.

In [ ]:
exp(-1.8 * 10)

**The other choices.**
- (A) `True False True`: takes the third `path(rng)` for the start of the stream of `default_rng(5)`,
  forgetting the two calls before it.
- (C) `True True True`: takes the simulator to copy the generator it is given.
- (D) `False False False`: takes an integer seed to draw fresh entropy at every call.

## 6.10 · `question.transposedKernelChangesTheRate`

The records of the asymmetric pair, kernel $(1.2, 4.8, 0.4, 0.8)$ in the order
(Component, Cause) $= (0,0), (0,1), (1,0), (1,1)$;
$\mu = (0.2, 0.6)$, $\beta = 4$; over $2\times10^4$ s, with standard errors below $0.04$.

**Answer: (B).** A number close to 3, then a number close to 2.

**The two matrices.**
`reshape(2, 2)` fills rows first, so row $e$ holds the records with `Component` $e$:
it is $A$, row excited and column exciting, as Remark 1.3.7 (`remark.indexOrder`) requires.
Its transpose exchanges $A_{0,1} = 4.8$ and $A_{1,0} = 0.4$.

**Stability.**
$\Gamma = A/4 = \begin{pmatrix} 0.3 & 1.2 \\ 0.1 & 0.2 \end{pmatrix}$
has trace $0.5$ and determinant $0.06 - 0.12 = -0.06$,
so its eigenvalues solve $x^2 - 0.5x - 0.06 = 0$: they are $0.6$ and $-0.1$, and $\rho = 0.6$.
$\Gamma^\top$ has the same eigenvalues.
Both satisfy the condition $\rho < 1$ of Theorem 1.3.19 (`thm.stationarity`).

In [ ]:
trace, determinant = 0.3 + 0.2, 0.3 * 0.2 - 1.2 * 0.1
(trace + sqrt(trace**2 - 4 * determinant)) / 2, (trace - sqrt(trace**2 - 4 * determinant)) / 2

**The rates.**
By eq. (1.3.18, `eq.stationaryIntensity`), with $\det(I - \Gamma) = 0.7 \cdot 0.8 - 1.2 \cdot 0.1 = 0.44$,
$$\lambda^* = \frac{1}{0.44}\begin{pmatrix} 0.8 & 1.2 \\ 0.1 & 0.7 \end{pmatrix}
\begin{pmatrix} 0.2 \\ 0.6 \end{pmatrix} = \begin{pmatrix} 2 \\ 1 \end{pmatrix},
\qquad
(I - \Gamma^\top)^{-1}\mu = \frac{1}{0.44}\begin{pmatrix} 0.8 & 0.1 \\ 1.2 & 0.7 \end{pmatrix}
\begin{pmatrix} 0.2 \\ 0.6 \end{pmatrix} = \begin{pmatrix} 0.5 \\ 1.5 \end{pmatrix}.$$
The total rates are 3 and 2, which differ by more than 25 standard errors.

The column sums of $\Gamma$, $0.4$ and $1.4$, and of $\Gamma^\top$, $1.5$ and $0.3$, are not constant:
F3's shortcut applies to neither matrix.
For the transpose $\mu$ is a right Perron vector, $\Gamma^\top\mu = 0.6\,\mu$,
so $(I - \Gamma^\top)^{-1}\mu = \mu/(1-\rho)$,
the second of the sufficient conditions stated after Corollary 1.3.25 (`corol.constantClusterSize`),
and its rate is $\bar\mu/(1-\rho) = 0.8/0.4 = 2$.

Remark 1.3.7 (`remark.indexOrder`) states that a transposed kernel passes every stability test
and, when $\mu$ is solved from a target $\lambda^*$, gives a path of the right total rate.
At a fixed $\mu$, as here, the rate changes.

In [ ]:
det = 0.7 * 0.8 - 1.2 * 0.1
lam = ((0.8 * 0.2 + 1.2 * 0.6) / det, (0.1 * 0.2 + 0.7 * 0.6) / det)
lam_t = ((0.8 * 0.2 + 0.1 * 0.6) / det, (1.2 * 0.2 + 0.7 * 0.6) / det)
det, lam, sum(lam), lam_t, sum(lam_t)

**The other choices.**
- (A) Close to 3 twice: reads the right total rate of Remark 1.3.7 (`remark.indexOrder`),
  which concerns a baseline solved from a target $\lambda^*$, as holding at a fixed $\mu$.
  Here $\mu$ is fixed; solved for the transpose from $\lambda^* = (2, 1)$ it would be $(1.3, -1.6)$, not a baseline.
- (C) Close to 2, then close to 3: reads `reshape` as filling columns first.
- (D) Close to 2, twice: F3's $\bar\mu/(1-\rho) = 0.8/0.4$ applied to both matrices;
  it holds for the transpose, of which $\mu$ is a right Perron vector, and not for $A$.